# Lab 6: Credit scores and lending decisions

A historical benchmark, honest validation and explicit costs

> **Note**
>
> **Time:** 100 minutes for the core lab. The two extensions can follow
> as independent practice.
>
> Attempt each question before revealing its sample answer. Sample
> answers appear on this HTML page and are omitted from the student
> notebook.
>
> <figure>
> <a
> href="https://colab.research.google.com/github/quinfer/fin510-colab-notebooks/blob/main/labs/lab06_alt_finance.ipynb"><img
> src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
> <figcaption>Open in Colab</figcaption>
> </figure>

A risk score ranks applications by estimated repayment risk. A lending
decision also needs a probability for the intended population, the
amount at risk, loan terms and the costs of different errors. Here we
compare two models on historical German credit records. We then use a
separate, hypothetical one-year contract to connect default
probabilities to cash flows. Read the
[chapter](https://financialdatascience.co.uk/chapters/05_alt_finance_marketplace_lending.html)
alongside the code.

## 1. Audit the target and the file

The UCI Statlog German Credit file contains 1,000 historical credit
records, each with 20 inputs. Its target is a good-credit or bad-credit
label: we code bad credit as 1 and good credit as 0. A predicted
probability therefore concerns the bad-credit label; its default horizon
is not established. UCI’s [corrected South German
documentation](https://archive.ics.uci.edu/dataset/522/south+german+credit)
dates the underlying credits to 1973–1975, describes heavy oversampling
of bad credits, and warns of errors in the original coding information
(Hofmann 1994; Grömping 2019). Treat the 30% bad-label share as a
sampling fact. The file does not supply an audited default definition, a
common outcome horizon, recoveries or dated cash flows.

The download is pinned to a repository commit and checked against the
retained CSV’s SHA-256 digest. Local files are checked too, so a
different dataset cannot silently inherit the displayed results. The
stored CSV column `defaulted` is a legacy name. We rename it
`bad_credit` on loading, without changing the stored data or pretending
to establish timed defaults. The credit-amount documentation also warns
of an unknown monotonic transformation; do not interpret it as audited
cash exposure. Categories remain opaque codes where their historical
meaning is disputed. In particular, personal status cannot be used to
reconstruct reliable sex labels.

In [ ]:
from pathlib import Path
import hashlib
from io import BytesIO
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (roc_auc_score, log_loss, brier_score_loss,
                             confusion_matrix, precision_recall_curve)
from sklearn.calibration import calibration_curve

paths = [Path("data/alt_finance/german_credit.csv"),
         Path("../data/alt_finance/german_credit.csv"),
         Path("../../data/alt_finance/german_credit.csv"), Path("german_credit.csv")]
local = next((p for p in paths if p.is_file()), None)
source = str(local) if local else (
    "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
    "027c830b4041ca07e1b298377a32ab0666b272d2/labs/german_credit.csv")
raw_bytes = local.read_bytes() if local else urlopen(source, timeout=30).read()
expected_sha256 = "37ef283edd556d84793d24cc455b868e2deb2dfc7c437fc70f7825ec4b07a708"
if hashlib.sha256(raw_bytes).hexdigest() != expected_sha256:
    raise ValueError("Dataset version changed: check provenance before reusing these results")
df = pd.read_csv(BytesIO(raw_bytes)).rename(columns={"defaulted": "bad_credit"})
assert df.shape == (1000, 21) and df.bad_credit.value_counts().to_dict() == {0: 700, 1: 300}
assert not df.isna().any().any()
print("Source:", source, "| scikit-learn:", sklearn.__version__)
print("Rows/columns:", df.shape, "| benchmark bad-label share:", df.bad_credit.mean())
print("Parsed-data fingerprint:", hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()[:16])
X = df.drop(columns="bad_credit")
y = df.bad_credit
numeric = X.select_dtypes(include="number").columns.tolist()
categorical = X.select_dtypes(exclude="number").columns.tolist()
assert len(numeric) == 7 and len(categorical) == 13
print("Seven numerical inputs:", numeric)
print("Thirteen categorical inputs:", categorical)
print(pd.crosstab(df.duration, y).head())

**Q1.** Write a data specification covering row, label, horizon,
population and information available at prediction time. Which parts
cannot be recovered here? Why is this not P2P lending data?

## 2. Split raw rows and fit preprocessing inside folds

A future applicant must be transformed using rules learned from training
observations. Fitting a scaler on every row before splitting violates
that boundary even when it barely changes the displayed AUC. An encoder
also learns its category vocabulary from training. Keep both inside a
pipeline. Unknown categories then have an explicit handling policy
instead of requiring a look at future applications. Here
`handle_unknown="ignore"` produces zero indicators for an unseen
category; with an omitted reference category, those zeros are
indistinguishable from the reference. Monitor new categories rather than
interpreting that fallback as evidence of equivalent risk.

We use all seven numerical inputs, including residence duration, in the
numerical baseline. This treats several ordinal codes as equally spaced,
a simplifying modelling choice rather than a fact about their
measurement. The richer model adds the thirteen coded categoricals,
including sensitive personal-status and foreign-worker information. It
is a feature-set comparison in this benchmark, not an experiment
identifying the value of modern alternative financial data.

| Data split | Role in this lab |
|:-----------------------------------|:-----------------------------------|
| Training: 600 rows | Learn preprocessing and model coefficients; compare training folds |
| Validation: 200 rows | Choose the F1 rejection threshold |
| Final test: 200 rows | Evaluate predictions and decision rules after choices are fixed |

Both models use logistic regression. Its parameter `C` sets inverse
regularisation strength: a smaller value penalises coefficients more
strongly. We use `C=1` in both models so changing the inputs does not
also change this fitting choice.

In [ ]:
development, test = train_test_split(X.index, test_size=0.20, random_state=42, stratify=y)
train, validation = train_test_split(development, test_size=0.25, random_state=43,
                                   stratify=y.loc[development])
assert not (set(train) & set(test) or set(validation) & set(test) or set(train) & set(validation))

def make_model(include_categories):
    steps = [("numeric", StandardScaler(), numeric)]
    if include_categories:
        steps.append(("category", OneHotEncoder(drop="first", handle_unknown="ignore",
                                               sparse_output=False), categorical))
    return Pipeline([("prepare", ColumnTransformer(steps, remainder="drop")),
                     ("logit", LogisticRegression(C=1, max_iter=3000))])

models = {"Numerical": make_model(False), "All inputs": make_model(True)}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X.loc[train], y.loc[train], cv=cv, scoring="roc_auc")
    cv_results[name] = scores
    model.fit(X.loc[train], y.loc[train])
    print(name, "training-fold AUCs:", scores.round(3),
          "mean/SD:", np.round([scores.mean(), scores.std()], 3))
print("Training/validation/test sizes:", len(train), len(validation), len(test))

**Q2.** Which operations learn from data? Explain why each must be
fitted within the training folds. Why keep `C=1` fixed in both models?
Suggest an alternative encoding for ordinal predictors.

## 3. Fix the decision rule before the test

Positive means flagging a bad credit risk. At a fixed score, lowering
the rejection threshold rejects more good risks and misses fewer bad
ones. The default cut 0.50 is a convention, not a credit policy. We
compare it with a threshold selected for F1 on validation rows, and with
the illustrative five-to-one error-cost cut $1/6$.

| Measure | Meaning when bad credit is the positive label |
|:-----------------------------------|:-----------------------------------|
| Precision | Correct bad-credit flags divided by all bad-credit flags |
| Recall | Correct bad-credit flags divided by all records labelled bad credit |
| F1 | Harmonic mean of precision and recall; it ignores correctly unflagged good-credit records |

F1 does not include loan amounts, interest, recoveries or fees. Let p be
predicted bad-credit probability. Approving has expected label-error
cost $5p$, and rejecting has expected cost $1-p$. Reject when $5p>1-p$,
which gives $p>1/6$; the two choices have equal cost at the boundary.
Here these are **illustrative label-error units within the sampled
benchmark**, not observed financial losses. The benchmark probabilities
are not population PDs, so neither cut is a deployment recommendation.
We freeze the rules before inspecting test labels.

In [ ]:
validation_probability = models["All inputs"].predict_proba(X.loc[validation])[:, 1]
pv, rv, thresholds = precision_recall_curve(y.loc[validation], validation_probability)
f1 = 2 * pv[:-1] * rv[:-1] / np.maximum(pv[:-1] + rv[:-1], 1e-12)
chosen_threshold = float(thresholds[np.argmax(f1)])
policies = {"Default 0.50": 0.50, "Validation F1": chosen_threshold,
            "Illustrative cost 1/6": 1 / 6}
print("Frozen policies:", policies)

**Q3.** Explain the access/capital trade-off using false positives and
false negatives. What would be wrong with maximising F1 on the test
labels and reporting that maximum as deployment performance?

## 4. Evaluate ranking and probability together

Use the final test only now. Compare the models on identical records.
AUC (area under the receiver operating characteristic curve) measures
how often the model gives a bad-credit record a higher risk score than a
good-credit record, with a tied pair counting as one half. Higher AUC
means better ranking. Lower Brier score and log-loss mean smaller
probability errors. Brier score and log-loss assess probability
performance, not calibration alone. AUC of 0.8 does not mean that 80% of
applicants were correctly classified or that 80% will repay. If these
results lead you to change the model or policy, the same test is no
longer fresh evaluation evidence; assess the revision on a new untouched
sample.

In [ ]:
y_test = y.loc[test].to_numpy()
probabilities = {name: model.predict_proba(X.loc[test])[:, 1] for name, model in models.items()}
metrics = pd.DataFrame({name: {
    "AUC": roc_auc_score(y_test, p), "log_loss": log_loss(y_test, p),
    "Brier": brier_score_loss(y_test, p)} for name, p in probabilities.items()}).T
print(metrics.round(4))
p = probabilities["All inputs"]
boot_rng = np.random.default_rng(2026)
paired_difference = []
for _ in range(2000):
    idx = boot_rng.integers(0, len(test), len(test))
    if np.unique(y_test[idx]).size == 2:
        paired_difference.append(roc_auc_score(y_test[idx], p[idx]) -
                                 roc_auc_score(y_test[idx], probabilities["Numerical"][idx]))
print("Paired AUC difference:", metrics.loc["All inputs", "AUC"] - metrics.loc["Numerical", "AUC"])
print("Conditional paired-bootstrap 95% interval:", np.quantile(paired_difference, [0.025, 0.975]))
observed, predicted = calibration_curve(y_test, p, n_bins=5, strategy="quantile")
counts = pd.Series(pd.qcut(p, 5, duplicates="drop")).value_counts(sort=False).to_numpy()
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.plot(predicted, observed, "o-", color="steelblue")
for a, b, count in zip(predicted, observed, counts):
    ax.annotate(f"n={count}", (a, b), xytext=(4, 5), textcoords="offset points")
ax.set(xlabel="Predicted benchmark bad-label probability", ylabel="Observed bad-label share",
       xlim=(0, 1), ylim=(0, 1))
plt.tight_layout()
plt.show()

**Q4.** Use the printed results to recommend a model within this
benchmark. What does the paired interval include, and what uncertainty
does it omit? Interpret a calibration point below the diagonal.

## 5. Count consequences at the fixed policies

Now measure what changes when a fixed score supports different
decisions. Keep the label-error cost separate from profit or fairness.
The cost cut may not have the lowest realised cost because probabilities
are approximate, sample sizes are small and its optimum is an
expectation under stated assumptions.

In [ ]:
policy_rows = []
for name, threshold in policies.items():
    flagged = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y_test, flagged).ravel()
    policy_rows.append({"policy": name, "threshold": threshold,
                        "approved": int((~flagged).sum()), "good_rejected": int(fp),
                        "bad_approved": int(fn), "precision": tp / max(tp + fp, 1),
                        "recall": tp / (tp + fn), "label_error_cost": int(fp + 5 * fn)})
policy_table = pd.DataFrame(policy_rows)
print(policy_table.round(3).to_string(index=False))

**Q5.** Which policy rejects most good risks? Which approves most bad
risks? Explain why the cheapest benchmark policy might still be
unsuitable for a lender. Write a three-sentence model recommendation
with one uncertainty and one deployment limitation.

## 6. A separate one-year cash-flow model

We now leave German Credit. Consider synthetic loans of £1, with assumed
probabilities of default (PDs), coupon interest rates and a common
one-year maturity. Coupon r is the contractual interest rate; p is the
assumed one-year probability of default. On repayment, the investor
receives principal plus coupon. On default, it recovers fraction $q$ of
principal and earns no coupon. Fee f is 1% of original principal and
applies to every loan at year end. Expected net return is
$(1-p)r-p(1-q)-f$. The numbers below are assumptions, not estimates from
the historical labels.

In [ ]:
contracts = pd.DataFrame({"scenario": ["Lower risk", "Intermediate", "Higher risk"],
                          "PD": [0.05, 0.15, 0.35], "coupon": [0.06, 0.15, 0.25]})
fee = 0.01
for recovery in [0, 0.40]:
    contracts[f"net_recovery_{recovery:.1f}"] = (
        (1 - contracts.PD) * contracts.coupon - contracts.PD * (1 - recovery) - fee)
print(contracts.round(4).to_string(index=False))
contracts.set_index("scenario").filter(like="net_").mul(100).plot.bar(figsize=(8, 4), rot=0)
plt.axhline(0, color="grey")
plt.ylabel("Expected one-year net return (%)")
plt.tight_layout()
plt.show()
print("Expected principal losses (£):", np.array([0.10, 0.05]) * np.array([100, 1000]))

**Q6.** At a 15% coupon and 1% fee, calculate break-even PD with zero
and 40% recovery. Why does higher AUC not automatically imply higher
investor returns? What changes for a multi-year amortising loan?

## 7. Extension: Change prevalence, preserve the assumptions

Prevalence is the proportion of records with the adverse label in a
population. Assume only this proportion changes, while predictor
distributions within each class stay fixed. A benchmark-calibrated
probability can then be adjusted through prior odds. We do not know the
historical population prevalence. Use a hypothetical 5% solely to assess
sensitivity.

In [ ]:
def prior_adjust(probability, sample_prevalence, target_prevalence):
    p = np.asarray(probability, dtype=float)
    if not (0 < sample_prevalence < 1 and 0 < target_prevalence < 1):
        raise ValueError("Prevalences must lie strictly between zero and one")
    if not np.all(np.isfinite(p) & (p >= 0) & (p <= 1)):
        raise ValueError("Probabilities must be finite and between zero and one")
    multiplier = (target_prevalence / (1 - target_prevalence)) / (
        sample_prevalence / (1 - sample_prevalence))
    return multiplier * p / (1 - p + multiplier * p)

print("Hypothetical 30% to 5% adjustment:", prior_adjust([0.10, 0.30, 0.60], 0.30, 0.05))
adjusted = prior_adjust(p, 0.30, 0.05)
print("Same-row AUC before/after monotone odds adjustment:",
      roc_auc_score(y_test, p), roc_auc_score(y_test, adjusted))
print("Flags at 1/6 before/after:", (p >= 1 / 6).sum(), (adjusted >= 1 / 6).sum())

**Q7.** Why does this transformation preserve ranking on these rows but
change decisions? Which assumptions would fail if loan approval selected
on income as well as risk? Does the transformed probability’s
calibration on this original sample establish target-population
calibration?

## 8. Extension: Group evaluation and monitoring

Use recorded age for a descriptive audit, with boundaries that match the
labels. Group sizes, label shares and policy errors answer different
questions. Neither a mean probability gap nor a coefficient identifies
unlawful discrimination. Sensitive coded fields elsewhere in this
benchmark are not a reliable basis for reconstructing sex groups.

In [ ]:
audit = X.loc[test, ["age"]].copy()
audit["bad_credit"] = y_test
audit["probability"] = p
audit["rejected"] = p >= 1 / 6
audit["group"] = pd.cut(audit.age, [0, 30, 50, np.inf], right=False,
                         labels=["Under 30", "30 to 49", "50 and over"])
group_rows = []
for name, group in audit.groupby("group", observed=True):
    bad = group.bad_credit == 1
    good = ~bad
    group_rows.append({"group": name, "n": len(group), "bad_n": int(bad.sum()),
                       "good_n": int(good.sum()), "bad_share": bad.mean(),
                       "mean_probability": group.probability.mean(),
                       "rejection_share": group.rejected.mean(),
                       "false_positive_rate": group.loc[good, "rejected"].mean(),
                       "false_negative_rate": (~group.loc[bad, "rejected"]).mean()})
print(pd.DataFrame(group_rows).round(3).to_string(index=False))

**Q8.** Distinguish a group calibration gap, a rejection-rate gap and an
error-rate gap. Which comparisons have small denominators? What would
you monitor prospectively if applicant mix changed before repayment
labels became available?

Grömping, Ulrike. 2019. “South German Credit.” UCI Machine Learning
Repository.
<https://archive.ics.uci.edu/dataset/522/south+german+credit>.

Hofmann, Hans. 1994. “Statlog (German Credit Data).” UCI Machine
Learning Repository. <https://doi.org/10.24432/C5NC77>.